# Analisis dan Pengujian FFNN
Dataset: **Global Student Placement & Salary Dataset**

Analisis ini meliputi pengujian hyperparameter, fungsi aktivasi, learning rate, regularisasi, dan perbandingan dengan library Scikit-Learn.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
import sys
import os
import seaborn as sns

sys.path.append(os.path.abspath('.'))
from ffnn import FFNN, Layer
import warnings
warnings.filterwarnings('ignore')

## 1. Data Loading & Preprocessing

In [ ]:
from pathlib import Path

project_root = Path.cwd().resolve()
dataset_candidates = [
    project_root / 'datasetml_2026.csv',
    project_root.parent / 'datasetml_2026.csv',
]

for candidate in dataset_candidates:
    if candidate.exists():
        dataset_path = candidate
        break
else:
    raise FileNotFoundError('datasetml_2026.csv not found in project root or parent of current directory.')

df = pd.read_csv(dataset_path)
print(f"Dataset loaded from: {dataset_path}")
print("Dataset shape:", df.shape)

# Target
y = df['placement_status'].apply(lambda x: 1 if x == 'Placed' else 0).values

# Features
X_raw = df.drop(columns=['placement_status'])
X_numeric = X_raw.select_dtypes(include=['float64', 'int64']).copy()
X_categorical = X_raw.select_dtypes(include=['object']).copy()

# One-hot encode categorical
X_cat_encoded = pd.get_dummies(X_categorical, drop_first=True)

# Combine features
X_combined = pd.concat([X_numeric, X_cat_encoded], axis=1).values

# Train test split
X_train, X_test, y_train, y_test = train_test_split(X_combined, y, test_size=0.2, random_state=42)

# StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

y_train = y_train.reshape(-1, 1)
y_test = y_test.reshape(-1, 1)

print(f"Train shapes: X={X_train.shape}, y={y_train.shape}")
print(f"Test shapes: X={X_test.shape}, y={y_test.shape}")



## Helper Functions
Fungsi untuk memplot *training* dan *validation loss* serta menampilkan metrik evaluasi.

In [ ]:
def plot_loss(history, title="Training History"):
    plt.figure(figsize=(8, 5))
    plt.plot(history['loss'], label='Train Loss')
    if 'val_loss' in history and len(history['val_loss']) > 0 and history['val_loss'][0] is not None:
        plt.plot(history['val_loss'], label='Val Loss')
    plt.title(title)
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid()
    plt.show()

def evaluate_model(y_true, y_pred, title="Evaluation Formulation"):
    y_pred_class = (y_pred >= 0.5).astype(int)
    acc = accuracy_score(y_true, y_pred_class)
    f1 = f1_score(y_true, y_pred_class, zero_division=0)
    print(f"--- {title} ---")
    print(f"Accuracy : {acc:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print("-" * (8 + len(title)))

## 2. Analisis Hyperparameter
### 2.a Pengaruh Depth & Width

Kita akan menguji 3 variasi kedalaman (depth: jumlah hidden layer = 1, 2, 3) dan 3 variasi kelebaran (width: 8, 16, 32 neuron).

In [ ]:
depths = [1, 2, 3]
widths = [8, 16, 32]
input_size = X_train.shape[1]

for d in depths:
    for w in widths:
        print(f"\nTraining model with Depth {d} & Width {w}")
        net = FFNN()
        
        # Input to first hidden
        net.add(Layer(input_size, w, activation='relu', weight_init='random_normal'))
        
        # Additional hidden layers
        for _ in range(d - 1):
            net.add(Layer(w, w, activation='relu', weight_init='random_normal'))
            
        # Output layer
        net.add(Layer(w, 1, activation='sigmoid', weight_init='random_normal'))
        
        net.compile(loss='binary_crossentropy')
        history = net.fit(X_train, y_train, epochs=20, batch_size=64, learning_rate=0.01, validation_data=(X_test, y_test), verbose=0)
        
        plot_loss(history, title=f"Depth {d}, Width {w}")
        y_pred = net.predict(X_test)
        evaluate_model(y_test, y_pred, f"Depth {d}, Width {w}")

### 2.b Pengaruh Fungsi Aktivasi Hidden Layer

Menguji ReLU, Sigmoid, dan Tanh pada hidden layer. (Softmax tidak diikutkan sesuai aturan)

In [ ]:
activations = ['relu', 'sigmoid', 'tanh']
input_size = X_train.shape[1]

for act in activations:
    print(f"\nTraining model with Activation {act}")
    net = FFNN()
    net.add(Layer(input_size, 16, activation=act, weight_init='random_normal'))
    net.add(Layer(16, 16, activation=act, weight_init='random_normal'))
    net.add(Layer(16, 1, activation='sigmoid', weight_init='random_normal'))
    
    net.compile(loss='binary_crossentropy')
    history = net.fit(X_train, y_train, epochs=20, batch_size=64, learning_rate=0.01, validation_data=(X_test, y_test), verbose=0)
    
    plot_loss(history, title=f"Activation {act}")
    y_pred = net.predict(X_test)
    evaluate_model(y_test, y_pred, f"Activation {act}")
    print(f"Plotting Weights and Gradients distribution ({act}):")
    net.plot_weight_distribution()
    net.plot_gradient_distribution()

### 2.c Pengaruh Learning Rate

Menguji Learning Rate: 0.1, 0.01, 0.001

In [ ]:
lrs = [0.1, 0.01, 0.001]
input_size = X_train.shape[1]

for lr in lrs:
    print(f"\nTraining model with Learning Rate {lr}")
    net = FFNN()
    net.add(Layer(input_size, 16, activation='relu', weight_init='random_normal'))
    net.add(Layer(16, 1, activation='sigmoid', weight_init='random_normal'))
    
    net.compile(loss='binary_crossentropy')
    history = net.fit(X_train, y_train, epochs=20, batch_size=64, learning_rate=lr, validation_data=(X_test, y_test), verbose=0)
    
    plot_loss(history, title=f"LR = {lr}")
    y_pred = net.predict(X_test)
    evaluate_model(y_test, y_pred, f"Learning Rate {lr}")
    print(f"Plotting Weights and Gradients distribution (LR = {lr}):")
    net.plot_weight_distribution()
    net.plot_gradient_distribution()

### 2.d Pengaruh Inisialisasi Bobot


In [ ]:
initializations = [
    ('zero', {}, 'Zero Initialization'),
    ('random_uniform', {'lower_bound': -0.3, 'upper_bound': 0.3, 'seed': 42}, 'Random Uniform Initialization'),
    ('random_normal', {'mean': 0.0, 'variance': 0.1, 'seed': 42}, 'Random Normal Initialization'),
]

input_size = X_train.shape[1]

for init_name, init_kwargs, label in initializations:
    print(f"\nTraining model with {label}")
    net = FFNN()
    net.add(Layer(input_size, 16, activation='relu', weight_init=init_name, **init_kwargs))
    net.add(Layer(16, 1, activation='sigmoid', weight_init=init_name, **init_kwargs))

    net.compile(loss='binary_crossentropy')
    history = net.fit(
        X_train,
        y_train,
        epochs=20,
        batch_size=64,
        learning_rate=0.01,
        validation_data=(X_test, y_test),
        verbose=0,
    )

    plot_loss(history, title=label)
    y_pred = net.predict(X_test)
    evaluate_model(y_test, y_pred, label)
    print(f"Plotting Weights and Gradients distribution ({label}):")
    net.plot_weight_distribution()
    net.plot_gradient_distribution()



## 3. Pengaruh Regularisasi

Membandingkan model tanpa regularisasi, dengan L1 (lambda 0.001), dan dengan L2 (lambda 0.001)

In [ ]:
regs = [(0, 0, "No Regularization"), (0.001, 0, "L1 Regularization"), (0, 0.001, "L2 Regularization")]
input_size = X_train.shape[1]

for l1, l2, name in regs:
    print(f"\nTraining model with {name}")
    net = FFNN()
    net.add(Layer(input_size, 32, activation='relu', weight_init='random_normal'))
    net.add(Layer(32, 16, activation='relu', weight_init='random_normal'))
    net.add(Layer(16, 1, activation='sigmoid', weight_init='random_normal'))
    
    net.compile(loss='binary_crossentropy', l1_lambda=l1, l2_lambda=l2)
    history = net.fit(X_train, y_train, epochs=30, batch_size=64, learning_rate=0.01, validation_data=(X_test, y_test), verbose=0)
    
    plot_loss(history, title=name)
    y_pred = net.predict(X_test)
    evaluate_model(y_test, y_pred, name)
    print(f"Plotting Weights and Gradients distribution ({name}):")
    net.plot_weight_distribution()
    net.plot_gradient_distribution()

## 4. Uji Perbandingan dengan Sklearn MLP

Membangun model from scratch terbaik vs `sklearn.neural_network.MLPClassifier`.

In [ ]:
# Final FFNN from Scratch
print("Training FFNN from Scratch...")
net_final = FFNN()
net_final.add(Layer(X_train.shape[1], 16, activation='relu', weight_init='random_normal'))
net_final.add(Layer(16, 1, activation='sigmoid', weight_init='random_normal'))
net_final.compile(loss='binary_crossentropy')
history_final = net_final.fit(X_train, y_train, epochs=50, batch_size=64, learning_rate=0.01, validation_data=(X_test, y_test), verbose=1)

plot_loss(history_final, title="FFNN From Scratch Loss")
y_pred_ffnn = net_final.predict(X_test)
evaluate_model(y_test, y_pred_ffnn, "FFNN From Scratch")

print("Plotting Weights and Gradients distribution (From Scratch):")
net_final.plot_weight_distribution()
net_final.plot_gradient_distribution()

print("\n" + "="*40 + "\n")

# Sklearn MLP
print("Training Sklearn MLPClassifier...")
mlp = MLPClassifier(hidden_layer_sizes=(16,), activation='relu', batch_size=64, learning_rate_init=0.01, max_iter=50, random_state=42)
mlp.fit(X_train, y_train.ravel())

plt.figure(figsize=(8, 5))
plt.plot(mlp.loss_curve_, label='Train Loss (Sklearn)')
plt.title("Sklearn MLPClassifier Loss")
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid()
plt.show()

y_pred_sklearn = mlp.predict(X_test)
evaluate_model(y_test, y_pred_sklearn, "Sklearn MLPClassifier")